In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from scipy import stats

In [ ]:
ghg_capita_df = pd.read_csv('data/EDGAR_GHG_per_capita.csv')
gain_df = pd.read_csv('data/gain.csv')
gdp_df = pd.read_csv('data/gdp_input.csv')

## EDA vor merge der Daten

Bevor die drei Datensätze (GHG, ND-GAIN, GDP) zusammenführen, prüfen:
1. **Keys / Duplikate**: Eindeutigkeit der Identifikatoren
2. **Year-Coverage**: Zeitliche Abdeckung
3. **Missingness**: Fehlende Werte pro Land
4. **Einheiten und Skalen**: Bedeutung der Werte
5. **ISO3 Konsistenz**: Ländercode-Kompatibilität

In [ ]:
ghg_capita_df.head()

In [ ]:
gain_df.head()

In [ ]:
gdp_df.head()

### 1. Datenstruktur & Identifikatoren

In [ ]:
# Shape und Spalten der drei Datensätze
print("GHG per Capita (EDGAR)")
print(f"Shape: {ghg_capita_df.shape}")
print(f"Spalten: {list(ghg_capita_df.columns[:5])}... (erste 5)")
print(f"\nIdentifikator-Spalten:")
for col in ['EDGAR Country Code', 'Country']:
    print(f"  - {col}: {ghg_capita_df[col].nunique()} unique values")

In [ ]:
print("ND-GAIN")
print(f"Shape: {gain_df.shape}")
print(f"Spalten: {list(gain_df.columns[:5])}... (erste 5)")
print(f"\nIdentifikator-Spalten:")
for col in ['ISO3', 'Name']:
    print(f"  - {col}: {gain_df[col].nunique()} unique values")

In [ ]:
print("GDP")
print(f"Shape: {gdp_df.shape}")
print(f"Spalten: {list(gdp_df.columns[:5])}... (erste 5)")
print(f"\nIdentifikator-Spalten:")
for col in ['ISO3', 'Name']:
    print(f"  - {col}: {gdp_df[col].nunique()} unique values")

In [ ]:
# Prüfen auf Duplikate in den Identifier-Spalten
# GHG
ghg_dups = ghg_capita_df['EDGAR Country Code'].duplicated().sum()
print(f"GHG - Duplikate in 'EDGAR Country Code': {ghg_dups}")
if ghg_dups > 0:
    print(f"  Duplikate: {ghg_capita_df[ghg_capita_df['EDGAR Country Code'].duplicated(keep=False)][['EDGAR Country Code', 'Country']].head()}")

# ND-GAIN
gain_dups = gain_df['ISO3'].duplicated().sum()
print(f"ND-GAIN - Duplikate in 'ISO3': {gain_dups}")
if gain_dups > 0:
    print(f"  Duplikate: {gain_df[gain_df['ISO3'].duplicated(keep=False)][['ISO3', 'Name']].head()}")

# GDP
gdp_dups = gdp_df['ISO3'].duplicated().sum()
print(f"GDP - Duplikate in 'ISO3': {gdp_dups}")
if gdp_dups > 0:
    print(f"  Duplikate: {gdp_df[gdp_df['ISO3'].duplicated(keep=False)][['ISO3', 'Name']].head()}")

### 2. Zeitliche Abdeckung

In [ ]:
# Jahr-Spalten identifizieren (Spalten die wie "1995" aussehen)
def get_year_columns(df):
    """Findet alle Spalten die Jahr-Daten enthalten (z.B. '1995', '2000')"""
    year_cols = [c for c in df.columns if str(c).isdigit()]
    return sorted([int(y) for y in year_cols])

ghg_years = get_year_columns(ghg_capita_df)
gain_years = get_year_columns(gain_df)
gdp_years = get_year_columns(gdp_df)

# Überlappung
overlap = set(ghg_years) & set(gain_years) & set(gdp_years)
print(f"Überlappende Jahre (alle 3 Datensätze): {min(overlap) if overlap else 'N/A'} - {max(overlap) if overlap else 'N/A'}")
print(f"Anzahl überlappender Jahre: {len(overlap)}")
print(f"Jahre: {sorted(overlap)}")

### 3. ISO3-Code Konsistenz & Überlappung
Auch wenn es anders genannt wird, sind die EDGAR Country Codes vom ghg Datensatz auch ISO3-Codes.

In [ ]:
# ISO3-Code Überlappung zwischen den Datensätzen

ghg_codes = set(ghg_capita_df['EDGAR Country Code'].dropna().unique())
gain_codes = set(gain_df['ISO3'].dropna().unique())
gdp_codes = set(gdp_df['ISO3'].dropna().unique())

print(f"\nAnzahl Länder pro Datensatz:")
print(f"  GHG (EDGAR):  {len(ghg_codes)} Länder")
print(f"  ND-GAIN:      {len(gain_codes)} Länder")
print(f"  GDP:          {len(gdp_codes)} Länder")

# Überlappung
all_three = ghg_codes & gain_codes & gdp_codes
ghg_gain = ghg_codes & gain_codes
ghg_gdp = ghg_codes & gdp_codes
gain_gdp = gain_codes & gdp_codes

print(f"\nÜberlappung:")
print(f"  In ALLEN 3 Datensätzen:     {len(all_three)} Länder")
print(f"  In GHG + ND-GAIN:           {len(ghg_gain)} Länder")
print(f"  In GHG + GDP:               {len(ghg_gdp)} Länder")
print(f"  In ND-GAIN + GDP:           {len(gain_gdp)} Länder")

# Nur in einem Datensatz
only_ghg = ghg_codes - gain_codes - gdp_codes
only_gain = gain_codes - ghg_codes - gdp_codes
only_gdp = gdp_codes - ghg_codes - gain_codes
gain_not_ghg = gain_codes - ghg_codes

print(f"Nur in ND-GAIN:  {len(only_gain)} Länder")
print(f"Nur in GDP:      {len(only_gdp)} Länder")
print(f"Nur in GHG:      {len(only_ghg)} Länder")
print(only_ghg)
print(f"In ND-GAIN/GDP, aber nicht GHG:  {len(gain_not_ghg)} Länder")
print(gain_not_ghg)


Im ghg_capita sind gewisse Länder zusammengenommen. z.B. MNE und SRB sind in GHG als SCG zusammengenommen oder LIE gehört dort zu CHE. -> Allenfalls in ND-Gain und GDP Datensätzen diese Zeilen zusammenaddieren? (Achtung, so wird wahrscheinlich ND-Gain verzogen?)

In ghg_capita ausserdem noch EU27 und Global Total vor mergen entfernen.

### 4. Missingness (Fehlende Werte)

In [ ]:
# GHG
print("\nGHG per Capita (EDGAR):")
print(f"{ghg_capita_df.isnull().sum().sum()} fehlende Werte gefunden")

# ND-GAIN
print("\nND-GAIN:")
print(f"{gain_df.isnull().sum().sum()} fehlende Werte gefunden")

# GDP
print("\nGDP:")
print(f"{gdp_df.isnull().sum().sum()} fehlende Werte gefunden")

In [ ]:
# ND-GAIN: Zeilen mit mindestens einem fehlenden Wert
gain_missing_rows = gain_df[gain_df.isnull().any(axis=1)]
print(f"ND-GAIN - Zeilen mit fehlenden Werten: {len(gain_missing_rows)} von {len(gain_df)}")
display(gain_missing_rows)

print("\n" + "="*80 + "\n")

# GDP: Zeilen mit mindestens einem fehlenden Wert
gdp_missing_rows = gdp_df[gdp_df.isnull().any(axis=1)]
print(f"GDP - Zeilen mit fehlenden Werten: {len(gdp_missing_rows)} von {len(gdp_df)}")
display(gdp_missing_rows)

### 5. Bereinigung
1. **Saint Kitts und Nevis** wird verworfen, da dazu keine ND-GAIN Daten vorliegen
2. **Serbien** und **Montenegro** sind bei Treibhausgasemissionen zusammengefasst (SCG). Daher werden die beiden Länder in den anderen Datensätzen ebenfalls zusammengefasst (jeweils gemittelt).
3. Folgende Staaten haben keine ND-Gain Daten und sind gleichzeitig im Treibhausgasemissions-Datensatz mit einem grösseren Land zusammengefasst:
    - **Andorra (mit ESP)**
    - **Liechtenstein (mit CHE)**
    - **Monaco (mit FRA)**
    - **San Marino (mit ITA)**
    Diese Länder werden aus dem ND-GAIN Datensatz entfernt und im Treibhausgasemissions-Datensatz werden die Länder umbenannt, sodass sie nur nach den grösseren Ländern benannt sind (z.B. "Spanien und Andorra" wird zu "Spanien").
4. **EU27** und **Global Total** werden aus dem Treibhausgasemissions-Datensatz entfernt.
5. Bei **PRK** und **CUB** werden im GDP Datensatz NA's eingefügt.
6. Folgende Staaten werden aus ND-Gain Daten und GDP Daten gelöscht, da dazu keine GHG Daten vorhanden sind: **NRU, FSM, TUV, MHL**
7. Folgende Staaten werden entfernt, da sie nur in den GHG Daten enthalten sind: **VGB, HKG, NCL, PRI, PYF, MAC, SHN, CYM, GIB, SPM, COK, GLP, MTQ, BMU, GUF, FRO, TWN, ESH, TCA, ABW, FLK, ANT, GRL, AIA, REU**

In [ ]:
# KNA (Saint Kitts and Nevis) aus allen drei Datensätzen entfernen
ghg_capita_df = ghg_capita_df[ghg_capita_df['EDGAR Country Code'] != 'KNA']
gain_df = gain_df[gain_df['ISO3'] != 'KNA']
gdp_df = gdp_df[gdp_df['ISO3'] != 'KNA']

In [ ]:
# Serbien (SRB) und Montenegro (MNE) zu SCG zusammenfassen in gain_df und gdp_df
gain_df = gain_df.copy()
gdp_df = gdp_df.copy()

# Finde Jahr-Spalten (alle numerischen Spalten)
def get_year_columns(df):
    return [col for col in df.columns if str(col).isdigit()]

# GAIN DataFrame: SRB und MNE mitteln
srb_gain = gain_df[gain_df['ISO3'] == 'SRB']
mne_gain = gain_df[gain_df['ISO3'] == 'MNE']

year_cols = get_year_columns(gain_df)
scg_gain = srb_gain.copy().iloc[0:1]
scg_gain['ISO3'] = 'SCG'
scg_gain['Name'] = 'Serbia and Montenegro'

# Jahr-Werte Mitteln
for col in year_cols:
    scg_gain[col] = (srb_gain[col].values[0] + mne_gain[col].values[0]) / 2

# SRB und MNE entfernen, füge SCG hinzu
gain_df = gain_df[~gain_df['ISO3'].isin(['SRB', 'MNE'])]
gain_df = pd.concat([gain_df, scg_gain], ignore_index=True)

# GDP DataFrame: SRB und MNE mitteln
srb_gdp = gdp_df[gdp_df['ISO3'] == 'SRB']
mne_gdp = gdp_df[gdp_df['ISO3'] == 'MNE']

year_cols = get_year_columns(gdp_df)
scg_gdp = srb_gdp.copy().iloc[0:1]
scg_gdp['ISO3'] = 'SCG'
scg_gdp['Name'] = 'Serbia and Montenegro'

# Jahr-Werte Mitteln
for col in year_cols:
    scg_gdp[col] = (srb_gdp[col].values[0] + mne_gdp[col].values[0]) / 2

# SRB und MNE entfernen, füge SCG hinzu
gdp_df = gdp_df[~gdp_df['ISO3'].isin(['SRB', 'MNE'])]
gdp_df = pd.concat([gdp_df, scg_gdp], ignore_index=True)

In [ ]:
# AND, LIE, MCO, SMR aus gain_df und gdp_df löschen
countries_to_remove = ['AND', 'LIE', 'MCO', 'SMR']
gain_df = gain_df[~gain_df['ISO3'].isin(countries_to_remove)]
gdp_df = gdp_df[~gdp_df['ISO3'].isin(countries_to_remove)]

In [ ]:
# Länder im ghg_capita_df umbenennen
ghg_capita_df = ghg_capita_df.copy()
country_renames = {
    'France and Monaco': 'France',
    'Italy, San Marino and the Holy See': 'Italy',
    'Spain and Andorra': 'Spain',
    'Switzerland and Liechtenstein': 'Switzerland'
}

for old_name, new_name in country_renames.items():
    ghg_capita_df.loc[ghg_capita_df['Country'] == old_name, 'Country'] = new_name

In [ ]:
# EU27 und GLOBAL TOTAL aus ghg_capita_df entfernen
ghg_capita_df = ghg_capita_df[~ghg_capita_df['EDGAR Country Code'].isin(['EU27', 'GLOBAL TOTAL'])]

In [ ]:
# in gdp_df PRK und CUB auf NaN setzen
year_cols = get_year_columns(gdp_df)

for iso_code in ['PRK', 'CUB']:
    mask = gdp_df['ISO3'] == iso_code
    gdp_df.loc[mask, year_cols] = np.nan

In [ ]:
# NRU, FSM, TUV, MHL aus gain_df und gdp_df entfernen
countries_to_remove_no_ghg = ['NRU', 'FSM', 'TUV', 'MHL']
gain_df = gain_df[~gain_df['ISO3'].isin(countries_to_remove_no_ghg)]
gdp_df = gdp_df[~gdp_df['ISO3'].isin(countries_to_remove_no_ghg)]

In [ ]:
# Länder aus ghg_capita_df entfernen
countries_only_in_ghg = {'VGB', 'HKG', 'NCL', 'PRI', 'PYF', 'MAC', 'SHN', 'CYM', 'GIB', 'SPM', 'COK', 'GLP', 'MTQ', 'BMU', 'GUF', 'FRO', 'TWN', 'ESH', 'TCA', 'ABW', 'FLK', 'ANT', 'GRL', 'AIA', 'REU'}

ghg_capita_df = ghg_capita_df[~ghg_capita_df['EDGAR Country Code'].isin(countries_only_in_ghg)]

In [ ]:
print("Nach Bereinigung:")
print(f"GAIN: {len(gain_df)} Länder")
print(f"GDP: {len(gdp_df)} Länder")
print(f"GHG: {len(ghg_capita_df)} Länder")

### 6. Merge der Datensätze

In [ ]:
# Konvertiere alle DataFrames ins Long-Format
years_to_keep = [str(y) for y in range(1995, 2024)]  # 1995-2023

# GHG DataFrame ins Long-Format
ghg_long = ghg_capita_df.melt(
    id_vars=['EDGAR Country Code', 'Country'],
    value_vars=[col for col in ghg_capita_df.columns if col in years_to_keep],
    var_name='Year',
    value_name='GHG_per_capita'
)
ghg_long['Year'] = ghg_long['Year'].astype(int)
ghg_long = ghg_long.rename(columns={'EDGAR Country Code': 'ISO3'})

print(f"GHG Long-Format: {ghg_long.shape}")
print(ghg_long.head())
print()

# GAIN DataFrame ins Long-Format
gain_long = gain_df.melt(
    id_vars=['ISO3', 'Name'],
    value_vars=[col for col in gain_df.columns if col in years_to_keep],
    var_name='Year',
    value_name='ND_GAIN'
)
gain_long['Year'] = gain_long['Year'].astype(int)

print(f"GAIN Long-Format: {gain_long.shape}")
print(gain_long.head())
print()

# GDP DataFrame ins Long-Format
gdp_long = gdp_df.melt(
    id_vars=['ISO3', 'Name'],
    value_vars=[col for col in gdp_df.columns if col in years_to_keep],
    var_name='Year',
    value_name='GDP_per_capita'
)
gdp_long['Year'] = gdp_long['Year'].astype(int)

print(f"GDP Long-Format: {gdp_long.shape}")
print(gdp_long.head())

In [ ]:
# Merge der drei DataFrames
# Zuerst GHG mit GAIN left joinen (Ländernamen von GHG sind konsitenter)
merged_df = ghg_long.merge(
    gain_long[['ISO3', 'Year', 'ND_GAIN']], 
    on=['ISO3', 'Year'], 
    how='left'
)

print(f"Nach GHG + GAIN Merge: {merged_df.shape}")

# GDP left joinen
merged_df = merged_df.merge(
    gdp_long[['ISO3', 'Year', 'GDP_per_capita']], 
    on=['ISO3', 'Year'], 
    how='left'
)

print(f"Nach vollständigem Merge: {merged_df.shape}")
print(f"NaN in GDP_per_capita: {merged_df['GDP_per_capita'].isna().sum()}")

# Sortiere nach Land und Jahr
merged_df = merged_df.sort_values(['ISO3', 'Year']).reset_index(drop=True)

print("\nFinaler Datensatz:")
print(merged_df.head(10))
print(f"\nShape: {merged_df.shape}")

## Explorative Datenanalyse mit gemergten Daten

### 1. Verteilungen der Variablen

In [ ]:
# Histogramme und Boxplots für die drei Hauptvariablen
fig, axes = plt.subplots(3, 2, figsize=(14, 12))

# GHG per capita
axes[0, 0].hist(merged_df['GHG_per_capita'].dropna(), bins=50, edgecolor='black')
axes[0, 0].set_title('Verteilung: Treibhausgase pro Kopf')
axes[0, 0].set_xlabel('GHG per capita (t CO2eq)')
axes[0, 0].set_ylabel('Häufigkeit')

axes[0, 1].boxplot(merged_df['GHG_per_capita'].dropna(), vert=True)
axes[0, 1].set_title('Boxplot: Treibhausgase pro Kopf')
axes[0, 1].set_ylabel('GHG per capita (t CO2eq)')

# ND-GAIN
axes[1, 0].hist(merged_df['ND_GAIN'].dropna(), bins=50, edgecolor='black', color='orange')
axes[1, 0].set_title('Verteilung: ND-GAIN Score')
axes[1, 0].set_xlabel('ND-GAIN Score')
axes[1, 0].set_ylabel('Häufigkeit')

axes[1, 1].boxplot(merged_df['ND_GAIN'].dropna(), vert=True)
axes[1, 1].set_title('Boxplot: ND-GAIN Score')
axes[1, 1].set_ylabel('ND-GAIN Score')

# GDP per capita
axes[2, 0].hist(merged_df['GDP_per_capita'].dropna(), bins=50, edgecolor='black', color='green')
axes[2, 0].set_title('Verteilung: BIP pro Kopf')
axes[2, 0].set_xlabel('GDP per capita (USD)')
axes[2, 0].set_ylabel('Häufigkeit')

axes[2, 1].boxplot(merged_df['GDP_per_capita'].dropna(), vert=True)
axes[2, 1].set_title('Boxplot: BIP pro Kopf')
axes[2, 1].set_ylabel('GDP per capita (USD)')

plt.tight_layout()
plt.show()

### 2. Zeitliche Entwicklung

In [ ]:
# Durchschnittliche Entwicklung über die Zeit
yearly_avg = merged_df.groupby('Year')[['GHG_per_capita', 'ND_GAIN', 'GDP_per_capita']].mean()

fig, axes = plt.subplots(3, 1, figsize=(14, 10))

# GHG Entwicklung
axes[0].plot(yearly_avg.index, yearly_avg['GHG_per_capita'], marker='o', linewidth=2, color='steelblue')
axes[0].set_title('Durchschnittliche Treibhausgasemissionen pro Kopf über Zeit', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Jahr')
axes[0].set_ylabel('GHG per capita (t CO2eq)')
axes[0].grid(True, alpha=0.3)

# ND-GAIN Entwicklung
axes[1].plot(yearly_avg.index, yearly_avg['ND_GAIN'], marker='o', linewidth=2, color='orange')
axes[1].set_title('Durchschnittlicher ND-GAIN Score über Zeit', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Jahr')
axes[1].set_ylabel('ND-GAIN Score')
axes[1].grid(True, alpha=0.3)

# GDP Entwicklung
axes[2].plot(yearly_avg.index, yearly_avg['GDP_per_capita'], marker='o', linewidth=2, color='green')
axes[2].set_title('Durchschnittliches BIP pro Kopf über Zeit', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Jahr')
axes[2].set_ylabel('GDP per capita (USD)')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Top 10 Länder höchsten/tiefsten durchschnittlichen Werten (über alle Jahre Gruppiert pro Land)
country_avg = merged_df.groupby(['ISO3', 'Country'])[['GHG_per_capita', 'ND_GAIN', 'GDP_per_capita']].mean()

print("10 Länder mit höchsten GHG-Ausstoss:")
top_ghg = country_avg.nlargest(10, 'GHG_per_capita')[['GHG_per_capita']]
print(top_ghg)

print("\n10 Länder mit niedrigstem GHG-Ausstoss:")
bottom_ghg = country_avg.nsmallest(10, 'GHG_per_capita')[['GHG_per_capita']]
print(bottom_ghg)

In [ ]:
print("10 Länder mit höchsten ND-GAIN Score (besser vorbereitet):")
top_gain = country_avg.nlargest(10, 'ND_GAIN')[['ND_GAIN']]
print(top_gain)

print("\n10 Länder mit niedrigstem ND-GAIN Score (höchstes Risiko):")
bottom_gain = country_avg.nsmallest(10, 'ND_GAIN')[['ND_GAIN']]
print(bottom_gain)

In [ ]:
print("\n10 Länder mit höchstem BIP pro Kopf:")
top_gdp = country_avg.nlargest(10, 'GDP_per_capita')[['GDP_per_capita']]
print(top_gdp)

print("\n10 Länder mit niedrigstem BIP pro Kopf:")
bottom_gdp = country_avg.nsmallest(10, 'GDP_per_capita')[['GDP_per_capita']]
print(bottom_gdp)

### 3. Korrelationsanalyse

In [ ]:
# Korrelationsmatrix
correlation_matrix = merged_df[['GHG_per_capita', 'ND_GAIN', 'GDP_per_capita']].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, 
            square=True, linewidths=1, fmt='.3f', vmin=-1, vmax=1)
plt.title('Korrelationsmatrix: GHG, ND-GAIN, GDP', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Scatter Plots für paarweise Beziehungen
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# GHG vs ND-GAIN
axes[0].scatter(merged_df['GHG_per_capita'], merged_df['ND_GAIN'], alpha=0.3, s=10)
axes[0].set_xlabel('GHG per capita (t CO2eq)')
axes[0].set_ylabel('ND-GAIN Score')
axes[0].set_title('GHG vs ND-GAIN')
axes[0].grid(True, alpha=0.3)

# GDP vs ND-GAIN
axes[1].scatter(merged_df['GDP_per_capita'], merged_df['ND_GAIN'], alpha=0.3, s=10, color='orange')
axes[1].set_xlabel('GDP per capita (USD)')
axes[1].set_ylabel('ND-GAIN Score')
axes[1].set_title('GDP vs ND-GAIN')
axes[1].grid(True, alpha=0.3)

# GDP vs GHG
axes[2].scatter(merged_df['GDP_per_capita'], merged_df['GHG_per_capita'], alpha=0.3, s=10, color='green')
axes[2].set_xlabel('GDP per capita (USD)')
axes[2].set_ylabel('GHG per capita (t CO2eq)')
axes[2].set_title('GDP vs GHG')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 4. Länderspezifische Analysen

In [ ]:
# Vergleich ausgewählter Länder
# Top 10 tiefster ND-Gain
# countries_of_interest = ['TCD', 'CAF', 'GNB', 'NER', 'SDN', 'COD', 'SLE', 'ERI', 'AFG', 'MLI']
# Top 10 höchster ND-Gain
countries_of_interest = ['NOR', 'FIN', 'NZL', 'DNK', 'CHE', 'SWE', 'GBR', 'DEU', 'CAN', 'ISL']
# Top 10 tiefster GHG-Ausstoss
# countries_of_interest = []
# Top 10 höchster GHG-Ausstoss
# countries_of_interest = []

fig, axes = plt.subplots(3, 1, figsize=(14, 12))

for country in countries_of_interest:
    country_data = merged_df[merged_df['ISO3'] == country].sort_values('Year')
    
    if not country_data.empty:
        # GHG Entwicklung
        axes[0].plot(country_data['Year'], country_data['GHG_per_capita'], 
                    marker='o', label=country, linewidth=2)
        
        # ND-GAIN Entwicklung
        axes[1].plot(country_data['Year'], country_data['ND_GAIN'], 
                    marker='o', label=country, linewidth=2)
        
        # GDP Entwicklung
        axes[2].plot(country_data['Year'], country_data['GDP_per_capita'], 
                    marker='o', label=country, linewidth=2)

axes[0].set_title('Treibhausgasemissionen pro Kopf: Ländervergleich', fontsize=12, fontweight='bold')
axes[0].set_ylabel('GHG per capita (t CO2eq)')
axes[0].legend(loc='best')
axes[0].grid(True, alpha=0.3)

axes[1].set_title('ND-GAIN Score: Ländervergleich', fontsize=12, fontweight='bold')
axes[1].set_ylabel('ND-GAIN Score')
axes[1].legend(loc='best')
axes[1].grid(True, alpha=0.3)

axes[2].set_title('BIP pro Kopf: Ländervergleich', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Jahr')
axes[2].set_ylabel('GDP per capita (USD)')
axes[2].legend(loc='best')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 5. Klassifizierung der Länder

In [ ]:
# Klassifizierung der Länder nach Durchschnittswerten (über alle Jahre)
# Nutzen der country_avg Variable

# Kategorisierung nach Medianen
ghg_median = country_avg['GHG_per_capita'].median()
gain_median = country_avg['ND_GAIN'].median()

# 4 Kategorien:
# 1. Hohe GHG, Hohes Risiko (niedriges ND-GAIN)
# 2. Hohe GHG, Niedriges Risiko (hohes ND-GAIN)
# 3. Niedrige GHG, Hohes Risiko (niedriges ND-GAIN)
# 4. Niedrige GHG, Niedriges Risiko (hohes ND-GAIN)

country_avg['GHG_category'] = country_avg['GHG_per_capita'].apply(lambda x: 'Hoch' if x > ghg_median else 'Niedrig')
country_avg['Risk_category'] = country_avg['ND_GAIN'].apply(lambda x: 'Niedriges Risiko' if x > gain_median else 'Hohes Risiko')

# Kombinierte Kategorie
country_avg['Combined_category'] = country_avg['GHG_category'] + ' GHG, ' + country_avg['Risk_category']

print("=== LÄNDER-KLASSIFIZIERUNG ===\n")
for category in country_avg['Combined_category'].unique():
    countries_in_cat = country_avg[country_avg['Combined_category'] == category]
    print(f"\n{category}: {len(countries_in_cat)} Länder")
    print(countries_in_cat.head(10).index.get_level_values('Country').tolist())
    
# Visualisierung: 2D Plot mit Quadranten
plt.figure(figsize=(12, 8))

for category in country_avg['Combined_category'].unique():
    mask = country_avg['Combined_category'] == category
    plt.scatter(country_avg[mask]['GHG_per_capita'], 
               country_avg[mask]['ND_GAIN'],
               label=category, alpha=0.6, s=100)

# Mediane als Linien
plt.axvline(x=ghg_median, color='red', linestyle='--', linewidth=1, alpha=0.5)
plt.axhline(y=gain_median, color='red', linestyle='--', linewidth=1, alpha=0.5)

plt.xlabel('Durchschnittliche GHG per capita (t CO2eq)', fontsize=12)
plt.ylabel('Durchschnittlicher ND-GAIN Score', fontsize=12)
plt.title('Länder-Klassifizierung: GHG vs Klimarisiko', fontsize=14, fontweight='bold')
plt.legend(loc='best')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()